# Week 2, day 5 (morning) — Extra practice 05 SOLUTIONS: break, continue, pass   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Question 6 is the one that matters. Everything else on the sheet is drill.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 05 — break, continue, pass. Run this once.
log = [
    "INFO  service started",
    "DEBUG cache warm",
    "WARN  slow query 2.1s",
    "INFO  batch 1 ok",
    "ERROR disk full",
    "INFO  batch 2 ok",
    "ERROR connection lost",
]

numbers = [4, 7, 0, 9, 3, 0, 11, 6]
codes = {"AA": 1, "BB": 0, "CC": 7, "DD": 0}

print(len(log), "log lines,", len(numbers), "numbers")

### Question 1

First error, then all errors. -> `first error at line 4 -- ERROR disk full`, `read 5 of 7 lines`, then `2 errors in total`.

Two loops, two different questions. The first stops at line 4 having read
five of seven lines; the second must read all seven, because a count cannot
be known early.

That is the rule for `break`: it belongs in a loop that is looking for
**one** thing. A loop that is counting, totalling or collecting has to see
everything, and a `break` in one of those is a bug.

`enumerate` gives the line number for free, which is the first thing anyone
will ask for when you report an error.

In [ ]:
read = 0
for i, entry in enumerate(log):
    read = read + 1
    if entry.startswith("ERROR"):
        print("first error at line", i, "--", entry)
        break

print("read", read, "of", len(log), "lines")

errors = 0
for entry in log:
    if entry.startswith("ERROR"):
        errors = errors + 1
print(errors, "errors in total")

### Question 2

`for` … `else`, twice. -> `no fatal errors`, then `WARN  slow query 2.1s`.

The first search finds nothing, runs to the end, and the `else` fires. The
second breaks, so the `else` is skipped.

"Found nothing" is a real result and needs its own line of code. Without
the `else` the first loop would print nothing at all — and nothing is
indistinguishable from a cell that did not run, which is how a missing
alert becomes a missing incident.

In [ ]:
for entry in log:
    if entry.startswith("FATAL"):
        print(entry)
        break
else:
    print("no fatal errors")

for entry in log:
    if entry.startswith("WARN"):
        print(entry)
        break
else:
    print("no warnings")

### Question 3

Skipping zeros. -> `40 from 6 values; 2 skipped`, then `mean of non-zero: 6.666666666666667` and `mean of all: 5.0`.

One total, two means, and they differ by a third. Both are correct
divisions of 40 — by 6 and by 8 — and the code cannot tell you which one
was wanted.

That depends entirely on what a zero **means**. A genuine reading of zero
belongs in the average; a zero that stands for "no data" does not. Nothing
in `[4, 7, 0, 9, 3, 0, 11, 6]` says which it is, and no amount of Python
will find out.

Which is why both are printed with their denominators named. A mean
without its denominator is not a result.

In [ ]:
total = 0
kept = 0
skipped = 0

for n in numbers:
    if n == 0:
        skipped = skipped + 1
        continue
    total = total + n
    kept = kept + 1

print(total, "from", kept, "values;", skipped, "skipped")
print("mean of non-zero:", total / kept)
print("mean of all:     ", total / len(numbers))

### Question 4

Stopping at a threshold. -> the running total after each value, then `stopped at 23 after 5 of 8 values`.

The total crossed 20 on the fifth value and the loop stopped, leaving three
numbers unexamined.

Notice the final total is **23, not 20**. The test happens after the
addition, so the loop overshoots by whatever the last value was — the same
overshoot as extra practice 04 Q6. If you needed the last value that kept
the total *under* 20, this loop has already gone past it and you would have
to check before adding rather than after.

The zero at position 3 was added and changed nothing, which is why it still
cost a pass.

In [ ]:
running = 0
used = 0

for n in numbers:
    running = running + n
    used = used + 1
    print("after", n, "->", running)
    if running > 20:
        break

print("stopped at", running, "after", used, "of", len(numbers), "values")

### Question 5

`pass` as a placeholder. -> `AA 1`, `BB 0`, `CC 7`, `DD 0`, then `4 lines from 4 codes`.

All four printed. `pass` did nothing, execution carried straight on to the
`print` below it, and the `if` might as well not be there.

That is the correct use: the branch is a **placeholder**, marking a
decision you have not made yet, and Python needs a statement in it because
an empty block is a `SyntaxError`.

The risk is leaving it in. A `pass` that survives to production is a
question nobody answered — here, what a zero code means. Better to leave
a `TODO` comment beside it, as the solution does, than to let it read as
deliberate.

In [ ]:
lines = 0
for code, value in codes.items():
    if value == 0:
        pass          # TODO: decide what a zero means
    print(code, value)
    lines = lines + 1

print(lines, "lines from", len(codes), "codes")

### Question 6

The question this sheet exists for. -> comprehension `[4, 7, 9, 3, 11, 6] 6`; loop `[4, 7] 2`; `agree: False`.

The comprehension gave six values, the loop gave two, and only the loop
answered the question that was asked.

**A comprehension has no `break`.** Its filter `if` decides whether each
item is kept — that is a `continue`, applied to every item, always. There
is no way to make a comprehension stop early, because it is defined as
visiting the whole collection.

So the rule is simple: if you need to **stop**, you need a loop. "Up to
the first X" and "everything except X" look almost the same in English and
are completely different in code, and this is the pair to remember them by.

(There is a way — `itertools.takewhile` — but it is a different tool, not a
comprehension, and it is not in this course.)

In [ ]:
attempt = [n for n in numbers if n != 0]
print("comprehension:", attempt, len(attempt))

correct = []
for n in numbers:
    if n == 0:
        break
    correct.append(n)

print("loop:         ", correct, len(correct))
print("agree:", attempt == correct)

# The filter `if` is a continue, not a break. A comprehension visits every
# item, always -- there is no way to make it stop early. If you need to
# stop, you need a loop.

### Question 7

All three together. -> `two errors -- stopping`, then `INFO 3, WARN 1, ERROR 2`, then `read 7 of 7 -- 0 never reached`.

The second `ERROR` is the last line of the log, so the `break` fired on the
final pass and nothing was actually skipped. `0 never reached` — which is
luck, not design. Put another line after it and the counts would be short,
with no sign of it in the output except that number.

That is why the line is printed. A loop that can stop early has to report
how much it looked at, or its counts are unbounded from below.

The `continue` for `DEBUG` still costs a pass — it is counted in `read`
before it is skipped. Skipping a row is not the same as not reading it.

In [ ]:
info = 0
warn = 0
error = 0
read = 0

for entry in log:
    read = read + 1
    if entry.startswith("DEBUG"):
        continue
    if entry.startswith("INFO"):
        info = info + 1
    elif entry.startswith("WARN"):
        warn = warn + 1
    elif entry.startswith("ERROR"):
        error = error + 1
        if error == 2:
            print("two errors -- stopping")
            break

print(f"INFO {info}, WARN {warn}, ERROR {error}")
print("read", read, "of", len(log), "--", len(log) - read, "never reached")

### Question 8

Nested search, twice. -> without the flag: three matches and `inner ran 16 times`. With it: `BB matched 0` and `inner ran 11 times`.

`AA` has value 1 and there is no 1 in `numbers`, so its inner loop ran all
eight passes and found nothing — that is where most of the 16 went.

The flag version stops at the first match: 8 fruitless passes for `AA`,
then 3 to reach the `0` for `BB`, then out of both loops. 11 instead of 16.

`break` leaves **one** loop. Getting out of two needs the flag, checked
immediately after the inner loop — worksheet 05 Q9 is the same pattern.
And the flag is doing double duty again: it exits the outer loop *and*
records whether anything was found at all.

In [ ]:
inner = 0
for code, value in codes.items():
    for n in numbers:
        inner = inner + 1
        if n == value:
            print(code, "matched", n)
            break
print("inner ran", inner, "times")

print("---")

inner = 0
matched = False
for code, value in codes.items():
    for n in numbers:
        inner = inner + 1
        if n == value:
            print(code, "matched", n)
            matched = True
            break
    if matched:
        break
print("inner ran", inner, "times")

### Question 9

The variable that was never assigned. -> `NameError: name 'first_error' is not defined`.

The loop worked perfectly. It found the error and broke out — and never
put it anywhere, so there is nothing to print.

The fix is two lines and people write one of them: `first_error = entry`
inside the loop is the obvious half. `first_error = None` **before** the
loop is the half that gets left out, and it is the one that makes the
not-found case work — without it, a log with no errors raises the same
`NameError` all over again.

**Initialise before the loop, assign inside it, read after it.** Same rule
as every accumulator on these sheets, and the same failure as worksheet 05
Q11.

In [ ]:
for entry in log:
    if entry.startswith("ERROR"):
        break

# This is SUPPOSED to raise: NameError. The loop found the error and broke
# out, but never assigned it to anything.
#
# `first_error = None` before the loop and `first_error = entry` inside it.
# The first of those two lines is the one people forget, and it is the one
# that makes the not-found case work as well.
print(first_error)